In [1]:
from pyspark.sql import SparkSession

spark = (
     SparkSession
    .builder
    .appName("monitor-spark-ui")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/01 11:55:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.sparkContext.setLogLevel("ERROR")

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", DateType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True)
])

In [4]:
schema

StructType([StructField('show_id', StringType(), True), StructField('type', StringType(), True), StructField('title', StringType(), True), StructField('director', StringType(), True), StructField('cast', StringType(), True), StructField('country', StringType(), True), StructField('date_added', DateType(), True), StructField('release_year', IntegerType(), True), StructField('rating', StringType(), True), StructField('duration', StringType(), True), StructField('listed_in', StringType(), True), StructField('description', StringType(), True)])

In [5]:
df = (
     spark.read.format("csv")
    .option("header", "true")
    .schema(schema)
    .load("../data/netflix_titles.csv")
)

In [6]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [7]:
df = df.filter(
    df.release_year > 2020
)

In [8]:
df = df.groupBy("country").count()

In [9]:
df.show()

+--------------------+-----+
|             country|count|
+--------------------+-----+
|India, United Kin...|    1|
|France, United St...|    3|
|              Sweden|    3|
|              Turkey|    5|
|China, United Sta...|    1|
|             Germany|    5|
|              Jordan|    1|
|              France|    7|
|    Uruguay, Germany|    1|
|United States, India|    1|
|Belgium, United K...|    1|
|                null|  208|
|           Argentina|    2|
|Mexico, United St...|    1|
|             Belgium|    2|
|               India|   31|
|       United States|  137|
|               China|    4|
|United States, Cz...|    1|
|United States, Japan|    2|
+--------------------+-----+
only showing top 20 rows



In [10]:
# spark.stop()